In [1]:
""""
Copyright (C) 2025 ETH Zurich, Stefan J.P. Chrin, Gregory Landrum, Sereina Riniker
"""

import pandas as pd
import itertools


# read data
df = pd.read_csv('../../../../0_DATA/HAHN/03b_all_ligands_all_ffs.csv')

# select relevant columns
fep_df = df[['ligand', 'target', 'DG_OPLS3e']]

print(len(df))
df.columns

598


Index(['lig', 'ligand', 'target', 'DG_Exp.', 'Ki', 'unit', 'dDG_Exp.',
       'DG_OpenFF-1.0', 'DG_OpenFF-1.0_converged', 'DG_OpenFF-1.2',
       'DG_OpenFF-1.2_converged', 'DG_OpenFF-2.0', 'DG_OpenFF-2.0_converged',
       'DG_OPLS3e', 'DG_GAFF2', 'DG_cGenFF',
       'DG_Consensus_OpenFF_GAFF2_cGenFF', 'DG_Consensus_OpenFF_GAFF2',
       'DG_Consensus_all', 'dDG_OpenFF-1.0', 'dDG_OpenFF-1.0_converged',
       'dDG_OpenFF-1.2', 'dDG_OpenFF-1.2_converged', 'dDG_OpenFF-2.0',
       'dDG_OpenFF-2.0_converged', 'dDG_OPLS3e', 'dDG_GAFF2', 'dDG_cGenFF',
       'dDG_Consensus_OpenFF_GAFF2_cGenFF', 'dDG_Consensus_OpenFF_GAFF2',
       'dDG_Consensus_all'],
      dtype='object')

### Get FEP dG values in the same structure (splits) as ML

In [2]:
import json

# ligand/pair ids repeat across targets, and the two tables use different
# target vocabularies, so matching must be on (target, name).
TARGET_NAME_MAP = {"hunt": "bace_hunt", "p2": "bace_p2"}


#Load file with the split structure
with open('../../../2D/RANDOM/A_RF_MORGAN_COUNT/dG_pred_&_error_all_splits.json', 'r') as f:
    splits = json.load(f)

# splits` is list of dicts
for split in splits:
    for target, ligand_dict in split.items():
        for ligand_name in sorted(ligand_dict):            # sorted for reproducibility
            ligand_name_clean = ligand_name.split("_", 1)[1] # remove lig from the start of the name as this is the format in 03b_all_ligands_all_ffs.csv
            row = fep_df[(fep_df['target'] == TARGET_NAME_MAP.get(target, target))
                         & (fep_df['ligand'] == ligand_name_clean)]
            if len(row) != 1:
                raise ValueError(f"Expected 1 FEP row for target '{target}' ligand '{ligand_name_clean}', got {len(row)}")
            # fetch DG and overwrite prediction
            dg_val = row['DG_OPLS3e'].iloc[0]
            ligand_dict[ligand_name]['prediction'] = dg_val

# save updated splits
with open('OPLS_RANDOM_dG_pred&error.json', 'w') as f:
    json.dump(splits, f, indent=2)